In [2]:
import os
import sys

# 1. Force the notebook to run from your project workspace root
project_root = r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression"
os.chdir(project_root)

# 2. Add the 'src' folder to Python's system path so it can see 'Linear_regression_01'
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Current Working Directory:", os.getcwd())
print("System path updated. Available modules:", os.listdir(src_path))

Current Working Directory: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression
System path updated. Available modules: ['Linear_regression_01', 'Linear_regression_01.egg-info']


In [3]:
import os
print(os.getcwd())

C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression


In [4]:
import sys
print(sys.path)

['c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire', '', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages', 'C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\linear_regression\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\pythonwin']


In [5]:
import os

print(os.path.exists("../src"))
print(os.path.exists("../src/Linear_regression_01"))

False
False


In [6]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(PROJECT_ROOT)
print(SRC_PATH)
print(sys.path[:3])

C:\Users\Greesha Vaishnavi\Desktop\dsprojects
C:\Users\Greesha Vaishnavi\Desktop\dsprojects\src
['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs']


In [7]:
import box
print(box.__version__)

7.4.1


In [8]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: Path
    unzip_data_dir: Path
    all_schema: dict

In [9]:
from Linear_regression_01.entity.config_entity import DataValidationConfig

In [10]:
# cofiguration manager
from Linear_regression_01.config.configuration import ConfigurationManager

def get_data_validation_config(self) -> DataValidationConfig:

    config = self.config.data_validation

    create_directories([config.root_dir])

    data_validation_config = DataValidationConfig(

        root_dir=config.root_dir,

        STATUS_FILE=config.STATUS_FILE,

        unzip_data_dir=config.unzip_data_dir,

        all_schema=config.all_schema

    )

    return data_validation_config

In [11]:
import pandas as pd
from Linear_regression_01.logging import logger

In [12]:
from Linear_regression_01.entity.config_entity import DataValidationConfig

In [ ]:
# Components 

class DataValidation:
    def __init__(self,config:DataValidationConfig):
        self.config = config

    def validate_dataset(self):

        if not os.path.exists(self.config.unzip_data_dir):  # func1 : Read datase
            logger.info("Dataset Not Found")
            return False
        df = pd.read_csv(self.config.unzip_data_dir)

        rows, columns = df.shape              #fun 2 : dataset shape 
        logger.info(f"Rows : {rows}")
        logger.info(f"Columns : {columns}")
        if rows == 0:
            validation_status = False

        expected_columns = list(expected_dtype.keys())
        actual_columns = list(df.columns)
        if expected_columns != actual_columns:            
            logger.info("Schema Validation Failed")
            validation_status = False
        else:
            logger.info("Schema Validation Passed")

        missing_values = df.isnull().sum()    # fun4 : find missing values
        logger.info(missing_values)
        if missing_values.sum()>0:
            validation_status=False

        duplicates = df.duplicated().sum()
        logger.info(f"Duplicate Rows : {duplicates}")  # fun5 : find duplicate values in rows 
        if duplicates>0:
            validation_status=False   
        logger.info(df.dtypes)       
  
        actual_dtype = df.dtypes.astype(str).to_dict()
        for column, dtype in expected_dtype.items():
         if actual_dtype[column] != dtype:
                logger.info(f"{column} datatype mismatch")
                validation_status = False

        if "price" not in df.columns:         #fun 6 target variable status check
            logger.info("Target Column Missing")
            validation_status = False

                                       # fun 7 invalid types yes and no in dataset  
        yes_no_columns = [

            "mainroad",

            "guestroom",

            "basement",

            "hotwaterheating",

            "airconditioning",

            "prefarea"

        ]

        for col in yes_no_columns:
            if not df[col].isin(["yes","no"]).all():
                logger.info(f"{col} contains invalid values")
                validation_status = False

        if not df["furnishingstatus"].isin( ["furnished","semi-furnished","unfurnished"]).all():  #its also invalid datatype                   
            logger.info("Invalid furnishingstatus values")
            validation_status = False

            
        numeric_columns = ["price",         # fun 8 negative numeric value
       "area",
       "bedrooms",
        "bathrooms",
        "stories",
        "parking"
     ]

        for col in numeric_columns:
            if (df[col] < 0).any():
                validation_status = False

        with open(self.config.STATUS_FILE,"w") as f:         #fun 9 write status and save
            f.write(f"Validation Status : {validation_status}")

        return validation_status     #fun 10 return status true/false

In [14]:
# pipeline 

STAGE_NAME="DATA VALIDATION STAGE"

class DataValidationTrainingPipeline:

    def main(self):

        config = ConfigurationManager()

        validation_config = config.get_data_validation_config()

        validation = DataValidation(validation_config)

        validation.validate_dataset()

In [15]:
obj = DataValidationTrainingPipeline()

obj.main()

[2026-07-24 07:30:51,858: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-24 07:30:51,863: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-24 07:30:51,866: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-07-24 07:30:51,866: INFO: common: created directory at artifacts]
[2026-07-24 07:30:51,866: INFO: common: created directory at artifacts/data_validation]
[2026-07-24 07:30:51,881: INFO: 3958141388: Rows : 545]
[2026-07-24 07:30:51,881: INFO: 3958141388: Columns : 13]
[2026-07-24 07:30:51,883: INFO: 3958141388: Schema Validation Failed]
[2026-07-24 07:30:51,885: INFO: 3958141388: price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64]


AttributeError: 'NoneType' object has no attribute 'items'